# 00 · Instalación y configuración
**Proyecto:** Influencia de los Sistemas Convectivos de Mesoescala (SCM) en la generación solar fotovoltaica del Magdalena
**Curso:** Análisis Geoespacial · UNAL Medellín · 2026-02

Este notebook se ejecuta **una sola vez** (o cada vez que cambies la configuración). Hace cuatro cosas:
1. Instala las librerías.
2. Crea `config.py` (período, dominio, umbrales) y `utils.py` (funciones compartidas).
3. Crea la tabla de parques `parques_magdalena.csv`.
4. Prueba las funciones con datos sintéticos, sin internet.

### Orden de los notebooks
| # | Notebook | Qué hace | Tiempo aprox. (1 mes) |
|---|---|---|---|
| 00 | Instalación y configuración | Este | 5 min |
| 01 | Límites y parques | Municipios del Magdalena y puntos de parques | 1 min |
| 02 | Modelo de elevación (DEM) | Copernicus 30 m, pendiente, DEM a 2 km | 5–10 min |
| 03 | Descarga GOES | Canal 13 (infrarrojo) y canal 2 (visible) | 1–3 h |
| 04 | Irradiancia satelital | GHI con índice de nubosidad | 5–10 min |
| 05 | Seguimiento de SCM | Identifica, sigue y clasifica los SCM | 10–20 min |
| 06 | Generación FV estimada | PVWatts en cada parque | 2 min |
| 07 | Generación real XM | Descarga la generación horaria de XM | 2–5 min |
| 08 | Análisis espacial | Mapas, Moran, Gi*, validación y pérdida real | 5 min |

### Dónde ejecutarlos
- **Jupyter local (Anaconda):** pon **todos los notebooks en la misma carpeta**. Ahí se crearán `config.py`, `utils.py` y las carpetas `datos/` y `resultados/`.
- **Google Colab:** la celda de preparación monta tu Drive y trabaja en `MyDrive/scm_magdalena`. Sube los 9 notebooks a esa carpeta y ábrelos desde ahí.

In [1]:
# ▶ CELDA DE PREPARACIÓN
import os, sys, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = Path("/content/drive/MyDrive/scm_magdalena")   # cámbiala si quieres
    CARPETA.mkdir(parents=True, exist_ok=True)
    os.chdir(CARPETA)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Carpeta de trabajo:", os.getcwd())

Carpeta de trabajo: c:\Users\User\Desktop\Unal\Analisis Geoespacial\notebooks_scm_magdalena


## 1. Instalar librerías
En **Windows con Anaconda**, si `geopandas` o `rasterio` fallan con pip, cierra Jupyter y en la terminal de Anaconda ejecuta:
```
conda install -c conda-forge geopandas rasterio pyproj netcdf4 h5netcdf
```
Luego vuelve a ejecutar esta celda.

In [2]:
PAQUETES = ["numpy", "scipy", "pandas", "xarray", "dask", "netCDF4", "h5netcdf", "s3fs", "pyproj", "shapely", "geopandas", "rasterio", "pvlib", "pydataxm", "libpysal", "esda", "matplotlib"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PAQUETES], check=True)
print("Listo. Si es la primera instalación, reinicia el kernel (Kernel → Restart) y sigue desde la celda de preparación.")

Listo. Si es la primera instalación, reinicia el kernel (Kernel → Restart) y sigue desde la celda de preparación.


## 2. Configuración del proyecto (`config.py`)
Aquí defines **todo lo que puedes cambiar**:
- **Período:** viene octubre de 2024 para probar. Para el trabajo final, cambia a 2025 completo.
- **Dominios:**
  - `DOM_SEGUIMIENTO` es amplio, para seguir los SCM desde que nacen.
  - `DOM_ANALISIS` es el Magdalena.
- **Umbrales de SCM:** temperatura de brillo, área y duración.
- **Parámetros del modelo fotovoltaico.**

Si modificas algo, vuelve a ejecutar esta celda y **reinicia el kernel** de los demás notebooks.

In [3]:
%%writefile config.py
"""
Configuración central del proyecto:
Influencia de los Sistemas Convectivos de Mesoescala (SCM) en la generación
solar fotovoltaica del departamento del Magdalena.

Edita SOLO este archivo para cambiar período, dominio o umbrales.
"""
from datetime import datetime
from pathlib import Path

# ----------------------------------------------------------------------------
# Rutas
# ----------------------------------------------------------------------------
BASE = Path(__file__).resolve().parent
DATA = BASE / "datos"
RAW = DATA / "crudos"          # descargas sin procesar
PROC = DATA / "procesados"     # productos intermedios
OUT = BASE / "resultados"      # mapas, tablas y figuras finales
for p in (DATA, RAW, PROC, OUT):
    p.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------------
# Período de estudio
# Recomendación: probar primero con UN mes y luego ampliar. Octubre de 2024 es
# temporada lluviosa en el Caribe y el parque Fundación ya estaba en operación.
# Para el trabajo final: 2025 completo.
# GOES-16 fue GOES-Este hasta el 2025-04-07; desde entonces es GOES-19
# (el script cambia de satélite automáticamente).
# ----------------------------------------------------------------------------
FECHA_INICIO = datetime(2024, 10, 1)
FECHA_FIN = datetime(2024, 10, 31)   # inclusive

# Horas UTC que se descargan
HORAS_TB = list(range(24))           # canal 13 (infrarrojo): 24 h, para seguir el ciclo de vida
HORAS_VIS = list(range(11, 24))      # canal 2 (visible): 11–23 UTC = 6–18 hora local

# ----------------------------------------------------------------------------
# Dominios (lat_min, lat_max, lon_min, lon_max)
# ----------------------------------------------------------------------------
# Dominio de seguimiento de SCM: más grande que el Magdalena para capturar
# sistemas que nacen en los piedemontes andinos, el valle del Magdalena,
# la Sierra Nevada o sobre el mar Caribe.
DOM_SEGUIMIENTO = (4.0, 15.0, -80.0, -68.0)
# Dominio de análisis: departamento del Magdalena con un pequeño margen.
DOM_ANALISIS = (8.6, 11.6, -75.2, -73.4)

# Región de estudio
REGION = "Magdalena"
COD_DANE = "47"              # código DANE del departamento

# ----------------------------------------------------------------------------
# Criterios de identificación de SCM (ajustables; ver README)
# Basados en la literatura de seguimiento con temperatura de brillo
# (p. ej. Feng et al., 2021; Zuluaga y Poveda, 2004).
# ----------------------------------------------------------------------------
TB_ESCUDO_K = 241.0        # escudo nuboso frío
TB_NUCLEO_K = 221.0        # núcleo convectivo profundo
AREA_MIN_KM2 = 5000.0      # área mínima del escudo en cada hora
AREA_MAX_MIN_KM2 = 10000.0 # el sistema debe alcanzar al menos esta área en su vida
DURACION_MIN_H = 3         # duración mínima (horas)
SOLAPE_MIN = 0.3           # fracción de solape para unir objetos entre horas

# ----------------------------------------------------------------------------
# Irradiancia satelital (método de índice de nubosidad tipo Heliosat)
# ----------------------------------------------------------------------------
PERCENTIL_CIELO_DESPEJADO = 5    # albedo de cielo despejado por píxel/hora/mes
PERCENTIL_NUBE = 99              # albedo de nube densa (todo el dominio, por mes)
ZENIT_MAX = 80.0                 # se descartan ángulos cenitales mayores

# ----------------------------------------------------------------------------
# Modelo fotovoltaico genérico (PVWatts) para los parques
# ----------------------------------------------------------------------------
INCLINACION = 10.0         # grados (latitud ~10 °N); cambia si conoces el diseño real del parque
AZIMUT = 180.0             # orientado al sur
GAMMA_PDC = -0.004         # coeficiente de temperatura (1/°C)
T_AIRE_C = 27.0            # temperatura del aire supuesta (mejorable con ERA5)
VIENTO_MS = 1.0
EFIC_INVERSOR = 0.96

# Resolución de la malla regular de salida (grados) ~2,2 km
RES_MALLA = 0.02

# Parques solares (editar/ampliar en parques_magdalena.csv)
ARCHIVO_PARQUES = BASE / "parques_magdalena.csv"

# Satélite GOES-Este según la fecha
CAMBIO_GOES19 = datetime(2025, 4, 7)

Writing config.py


## 3. Funciones compartidas (`utils.py`)
No necesitas modificarlas. Contienen:
- **Posición del sol:** `cos_zenit`.
- **Índice de nubosidad e índice de cielo despejado (Heliosat-2):** `indice_nubosidad`, `indice_cielo_despejado`.
- **Identificación y seguimiento de SCM:** `objetos_frios`, `Seguidor`, `etapa_ciclo_vida`.
- **Proyección de GOES y remuestreo a malla regular:** `indices_dominio`, `latlon_subconjunto`, `a_malla_regular`.

In [4]:
%%writefile utils.py
"""
Funciones compartidas. Las de las secciones 1–3 solo usan numpy/scipy,
así que se pueden probar sin descargar datos (ver test_utils.py).
"""
import numpy as np
from scipy import ndimage

R_TIERRA_KM = 6371.0


# ============================================================================
# 1. Posición solar (algoritmo de la NOAA, vectorizado; error < 0,5°)
# ============================================================================
def cos_zenit(tiempos_utc, lat, lon):
    """
    tiempos_utc: array de numpy datetime64 (n_t,)
    lat, lon: arrays 2D (ny, nx) en grados
    Devuelve cos(ángulo cenital) con forma (n_t, ny, nx).
    """
    t = np.asarray(tiempos_utc, dtype="datetime64[s]")
    doy = (t.astype("datetime64[D]") - t.astype("datetime64[Y]")).astype(int) + 1
    hora = (t - t.astype("datetime64[D]")).astype(float) / 3600.0
    g = 2 * np.pi / 365.0 * (doy - 1 + (hora - 12) / 24.0)
    ecuacion_tiempo = 229.18 * (0.000075 + 0.001868 * np.cos(g) - 0.032077 * np.sin(g)
                                - 0.014615 * np.cos(2 * g) - 0.040849 * np.sin(2 * g))
    decl = (0.006918 - 0.399912 * np.cos(g) + 0.070257 * np.sin(g)
            - 0.006758 * np.cos(2 * g) + 0.000907 * np.sin(2 * g)
            - 0.002697 * np.cos(3 * g) + 0.00148 * np.sin(3 * g))
    lat_r = np.deg2rad(lat)[None]
    # tiempo solar verdadero (min)
    tst = hora[:, None, None] * 60 + ecuacion_tiempo[:, None, None] + 4 * lon[None]
    ang_horario = np.deg2rad(tst / 4.0 - 180.0)
    d = decl[:, None, None]
    cz = np.sin(lat_r) * np.sin(d) + np.cos(lat_r) * np.cos(d) * np.cos(ang_horario)
    return np.clip(cz, -1, 1)


# ============================================================================
# 2. Índice de nubosidad e índice de cielo despejado (Heliosat-2, Rigollier 2004)
# ============================================================================
def indice_nubosidad(rho, rho_despejado, rho_nube):
    return (rho - rho_despejado) / np.maximum(rho_nube - rho_despejado, 1e-3)


def indice_cielo_despejado(n):
    """Relación empírica Kc(n) de Heliosat-2."""
    n = np.asarray(n, dtype=float)
    kc = np.where(n < -0.2, 1.2,
         np.where(n < 0.8, 1.0 - n,
         np.where(n < 1.1, 2.0667 - 3.6667 * n + 1.6667 * n ** 2, 0.05)))
    return np.where(np.isfinite(n), kc, np.nan)


# ============================================================================
# 3. Identificación y seguimiento de SCM sobre campos de temperatura de brillo
# ============================================================================
def area_pixel_km2(lat, lon):
    """Área aproximada de cada píxel de una malla 2D (lat, lon en grados)."""
    dlat = np.abs(np.gradient(lat, axis=0))
    dlon = np.abs(np.gradient(lon, axis=1))
    dy = np.deg2rad(dlat) * R_TIERRA_KM
    dx = np.deg2rad(dlon) * R_TIERRA_KM * np.cos(np.deg2rad(lat))
    return dx * dy


def objetos_frios(tb, area_pix, tb_escudo, tb_nucleo, area_min):
    """
    Etiqueta escudos nubosos fríos en un instante.
    Devuelve (etiquetas, lista de dicts con propiedades) solo para objetos que
    cumplen área mínima y tienen núcleo convectivo.
    """
    mascara = np.isfinite(tb) & (tb <= tb_escudo)
    etiquetas, n = ndimage.label(mascara, structure=np.ones((3, 3)))
    props = []
    salida = np.zeros_like(etiquetas)
    k = 0
    for i, sl in enumerate(ndimage.find_objects(etiquetas), start=1):
        if sl is None:
            continue
        m = etiquetas[sl] == i
        area = float(area_pix[sl][m].sum())
        tb_min = float(np.nanmin(tb[sl][m]))
        if area < area_min or tb_min > tb_nucleo:
            continue
        k += 1
        salida[sl][m] = k
        props.append({"obj": k, "area_km2": area, "tb_min_k": tb_min,
                      "tb_media_k": float(np.nanmean(tb[sl][m]))})
    return salida, props


class Seguidor:
    """
    Une objetos entre instantes consecutivos por solape espacial, hora a hora
    (guarda solo el instante previo, para no agotar la memoria).
    Regla: un objeto continúa la trayectoria del objeto previo más grande con el
    que se solapa (fracción respecto al menor de los dos >= solape_min). En una
    división, el trozo más grande conserva el id; en una fusión sobrevive el id
    del sistema previo más grande.
    """
    def __init__(self, solape_min):
        self.solape_min = solape_min
        self.siguiente_id = 1
        self.previo = None

    def paso(self, et):
        if et is None:
            self.previo = None
            return None
        previo = self.previo
        tr = np.zeros(et.shape, dtype="int32")
        usados = set()
        for obj in np.unique(et[et > 0]):
            m = et == obj
            asignado = None
            if previo is not None:
                ids_prev, cuentas = np.unique(previo[m & (previo > 0)], return_counts=True)
                mejor, mejor_tam = None, 0
                for idp, c in zip(ids_prev, cuentas):
                    tam_prev = int((previo == idp).sum())
                    if c / min(tam_prev, int(m.sum())) >= self.solape_min and tam_prev > mejor_tam:
                        mejor, mejor_tam = int(idp), tam_prev
                if mejor is not None and mejor not in usados:
                    asignado = mejor
                elif mejor is not None and m.sum() > (tr == mejor).sum():
                    tr[tr == mejor] = self.siguiente_id   # el trozo menor recibe id nuevo
                    usados.add(self.siguiente_id)
                    self.siguiente_id += 1
                    asignado = mejor
            if asignado is None:
                asignado = self.siguiente_id
                self.siguiente_id += 1
            usados.add(asignado)
            tr[m] = asignado
        self.previo = tr
        return tr


def seguir_objetos(lista_etiquetas, solape_min):
    """Versión por lotes (para pruebas): lista de etiquetas → lista de trayectorias."""
    s = Seguidor(solape_min)
    return [s.paso(et) for et in lista_etiquetas]


def etapa_ciclo_vida(areas):
    """
    Clasifica cada hora de una trayectoria según su área:
    'iniciación' antes del 80 % del área máxima, 'madurez' mientras el área
    está por encima del 80 % del máximo, 'disipación' después.
    """
    a = np.asarray(areas, dtype=float)
    umbral = 0.8 * a.max()
    sobre = np.where(a >= umbral)[0]
    i0, i1 = sobre.min(), sobre.max()
    etapas = np.array(["madurez"] * len(a), dtype=object)
    etapas[:i0] = "iniciación"
    etapas[i1 + 1:] = "disipación"
    return etapas.tolist()


# ============================================================================
# 4. Proyección de la malla fija de GOES (requiere pyproj)
# ============================================================================
def proyeccion_goes(ds):
    from pyproj import Proj
    p = ds["goes_imager_projection"].attrs
    h = float(p["perspective_point_height"])
    proj = Proj(proj="geos", h=h, lon_0=float(p["longitude_of_projection_origin"]),
                a=float(p["semi_major_axis"]), b=float(p["semi_minor_axis"]),
                sweep=str(p["sweep_angle_axis"]))
    return proj, h


def indices_dominio(ds, dominio, n=60):
    """Índices (y0, y1, x0, x1) de la malla fija que cubren el dominio lat/lon."""
    proj, h = proyeccion_goes(ds)
    la0, la1, lo0, lo1 = dominio
    lats = np.r_[np.linspace(la0, la1, n), np.linspace(la0, la1, n),
                 np.full(n, la0), np.full(n, la1)]
    lons = np.r_[np.full(n, lo0), np.full(n, lo1),
                 np.linspace(lo0, lo1, n), np.linspace(lo0, lo1, n)]
    xm, ym = proj(lons, lats)
    xr_, yr_ = np.asarray(xm) / h, np.asarray(ym) / h
    x = ds["x"].values
    y = ds["y"].values
    xi = np.where((x >= xr_.min()) & (x <= xr_.max()))[0]
    yi = np.where((y >= yr_.min()) & (y <= yr_.max()))[0]
    return int(yi.min()), int(yi.max()) + 1, int(xi.min()), int(xi.max()) + 1


def latlon_subconjunto(ds_sub):
    proj, h = proyeccion_goes(ds_sub)
    X, Y = np.meshgrid(ds_sub["x"].values * h, ds_sub["y"].values * h)
    lon, lat = proj(X, Y, inverse=True)
    lon = np.where(np.abs(lon) > 360, np.nan, lon)
    lat = np.where(np.abs(lat) > 90, np.nan, lat)
    return np.asarray(lat), np.asarray(lon)


def malla_regular(dominio, res):
    """Centros de píxel de la malla regular (norte→sur, oeste→este), igual a dem_2km.tif."""
    la0, la1, lo0, lo1 = dominio
    ny, nx = int(round((la1 - la0) / res)), int(round((lo1 - lo0) / res))
    return la1 - res * (np.arange(ny) + 0.5), lo0 + res * (np.arange(nx) + 0.5)


_cache_remuestreo = {}


def a_malla_regular(campo, lat, lon, dominio, res):
    """Remuestrea (vecino más cercano) un campo 2D de malla GOES a malla lat/lon regular.
    Los índices se calculan una sola vez por malla de origen y se reutilizan (rápido)."""
    from scipy.spatial import cKDTree
    lats_r, lons_r = malla_regular(dominio, res)
    clave = (lat.shape, float(np.nanmean(lat)), float(np.nanmean(lon)), tuple(dominio), res)
    if clave not in _cache_remuestreo:
        LO, LA = np.meshgrid(lons_r, lats_r)
        ok = np.isfinite(lat) & np.isfinite(lon)
        arbol = cKDTree(np.c_[lat[ok], lon[ok]])
        dist, idx = arbol.query(np.c_[LA.ravel(), LO.ravel()])
        _cache_remuestreo[clave] = (ok, idx, (dist > 2 * res).reshape(LA.shape), LA.shape)
    ok, idx, lejos, forma = _cache_remuestreo[clave]
    vals = np.asarray(campo)[ok][idx].reshape(forma).astype("float64")
    vals[lejos] = np.nan
    return vals, lats_r, lons_r

Writing utils.py


## 4. Parques solares (`parques_magdalena.csv`)
**Revisa las coordenadas en Google Earth** antes de seguir. Ubica el parque en la imagen satelital y copia lat/lon del centro de los paneles.

`clave_xm` es la palabra con la que se busca el parque en XM (notebook 07).

In [5]:
%%writefile parques_magdalena.csv
nombre,municipio,lat,lon,capacidad_mw,estado,clave_xm,coordenadas,fuente
Parque Solar Fundación (Enel),Fundación / Pivijay,10.4812,-74.2263,100,Operación comercial desde 2024 (despacho centralizado),FUNDACION,GEM (WGS84) - verificar con imagen satelital,Enel / Forbes / Global Energy Monitor
Caimán Cienaguero,Ciénaga,11.006,-74.247,14.3,Operación comercial desde ene-2024,CAIMAN,VERIFICAR - aproximada al casco urbano de Ciénaga,Energías Renovables (2024)

Writing parques_magdalena.csv


## 5. Prueba sin internet
Comprueba que las funciones hacen lo esperado con datos inventados:
- **El sol:** casi en el cenit al mediodía y bajo el horizonte de noche.
- **El índice de cielo despejado.**
- **Un SCM sintético:** crece, madura y se disipa mientras se desplaza, y debe quedar como **una sola trayectoria**.

In [6]:
import importlib, numpy as np
import utils; importlib.reload(utils)
from utils import cos_zenit, indice_cielo_despejado, area_pixel_km2, objetos_frios, seguir_objetos, etapa_ciclo_vida

# Posición solar en Santa Marta
lat = np.array([[11.24]]); lon = np.array([[-74.2]])
t = np.array(["2024-03-20T17:00", "2024-03-20T12:00", "2024-03-20T05:00"], dtype="datetime64[s]")
cz = cos_zenit(t, lat, lon)[:, 0, 0]
assert cz[0] > 0.95 and 0 < cz[1] < 0.35 and cz[2] < 0
print("Posición solar OK:", np.round(cz, 3), "(mediodía, amanecer, noche)")

# Índice de cielo despejado
kc = indice_cielo_despejado(np.array([-0.5, 0.0, 0.5, 1.5]))
assert np.allclose(kc, [1.2, 1.0, 0.5, 0.05])
print("Índice de cielo despejado OK:", kc)

# SCM sintético
ny, nx = 200, 300
LAT = np.repeat(np.linspace(14, 6, ny)[:, None], nx, axis=1)
LON = np.repeat(np.linspace(-80, -70, nx)[None, :], ny, axis=0)
apix = area_pixel_km2(LAT, LON)
yy, xx = np.mgrid[0:ny, 0:nx]
etiquetas = []
for k, r in enumerate([10, 20, 30, 30, 20, 8]):
    tb = np.full((ny, nx), 290.0)
    d = np.hypot(xx - (60 + 12 * k), yy - 100)
    tb[d < r] = 235.0; tb[d < r / 3] = 210.0
    tb[10:13, 250:253] = 215.0                    # nube pequeña que debe descartarse
    etiquetas.append(objetos_frios(tb, apix, 241, 221, 2000)[0])
tray = seguir_objetos(etiquetas, 0.3)
ids = {int(v) for tr in tray for v in np.unique(tr[tr > 0])}
areas = [float(apix[tr > 0].sum()) for tr in tray]
assert len(ids) == 1
print("Seguimiento OK: trayectorias =", ids)
print("Áreas (km²):", np.round(areas))
print("Etapas:", etapa_ciclo_vida(areas))
print("\n✅ Todo listo. Sigue con el notebook 01.")

Posición solar OK: [ 0.981  0.234 -0.981] (mediodía, amanecer, noche)
Índice de cielo despejado OK: [1.2  1.   0.5  0.05]
Seguimiento OK: trayectorias = {1}
Áreas (km²): [ 4994. 20383. 45988. 45988. 20383.  3160.]
Etapas: ['iniciación', 'iniciación', 'madurez', 'madurez', 'disipación', 'disipación']

✅ Todo listo. Sigue con el notebook 01.
